In [14]:
import requests
import pandas as pd
import numpy as np
import time
import os
import warnings
import statsmodels.api as sm
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
warnings.filterwarnings("ignore")
 

In [15]:
# ============================================================
# CONFIGURATION
# ============================================================
ARTICLES    = ["Bitcoin", "Ethereum", "Cryptocurrency"]
START_DATE  = "2019-01-01T00:00:00Z"
END_DATE    = "2023-12-31T23:59:59Z"
WIKI_FILE   = "wiki_edits.csv"
BTC_FILE    = "btc_prices.csv"
FINAL_FILE  = "wiki_tfidf_final.csv"

In [16]:
# ============================================================
# STEP 1: FETCH WIKIPEDIA REVISIONS
# ============================================================
print("=" * 60)
print("STEP 1: Fetching Wikipedia revisions")
print("=" * 60)
 
def fetch_revisions(article, start, end):
    url       = "https://en.wikipedia.org/w/api.php"
    headers   = {"User-Agent": "EconomicsResearch/1.0 (student research project)"}
    revisions = []
    rvcontinue = None
    page = 1
 
    print(f"\n  📖 Fetching: '{article}'")
 
    while True:
        params = {
            "action":  "query", "format": "json",
            "prop":    "revisions", "titles": article,
            "rvprop":  "ids|timestamp|user|comment|flags|size",
            "rvstart": end, "rvend": start,
            "rvlimit": "500", "rvdir": "older",
        }
        if rvcontinue:
            params["rvcontinue"] = rvcontinue
 
        for attempt in range(3):
            try:
                response = requests.get(url, params=params, headers=headers, timeout=30)
                data = response.json()
                break
            except Exception as e:
                print(f"    ⚠️ Attempt {attempt+1} failed: {e} — retrying in 5s...")
                time.sleep(5)
        else:
            print("    ❌ All retries failed, skipping page.")
            break
 
        pages = data.get("query", {}).get("pages", {})
        for page_id, page_data in pages.items():
            for rev in page_data.get("revisions", []):
                revisions.append({
                    "article":  article,
                    "rev_id":   rev.get("revid"),
                    "timestamp": rev.get("timestamp", "")[:10],
                    "user":     rev.get("user", ""),
                    "comment":  rev.get("comment", ""),
                    "is_minor": 1 if "minor" in rev else 0,
                    "is_anon":  1 if rev.get("anon") is not None else 0,
                    "size":     rev.get("size", 0),
                })
 
        print(f"    Page {page} — revisions so far: {len(revisions)}")
 
        if "continue" in data:
            rvcontinue = data["continue"].get("rvcontinue")
            page += 1
            time.sleep(0.5)
        else:
            print(f"    ✅ Done — {len(revisions)} revisions for '{article}'")
            break
 
    return revisions
 
if os.path.exists(WIKI_FILE):
    print(f"  📂 Found existing {WIKI_FILE} — skipping fetch.")
    wiki = pd.read_csv(WIKI_FILE)
else:
    all_revisions = []
    for article in ARTICLES:
        revs = fetch_revisions(article, START_DATE, END_DATE)
        all_revisions.extend(revs)
        time.sleep(1)
 
    wiki = pd.DataFrame(all_revisions)
    wiki["timestamp"] = pd.to_datetime(wiki["timestamp"])
    wiki = wiki.sort_values("timestamp").reset_index(drop=True)
    wiki["is_revert"] = wiki["comment"].str.lower().str.contains(
        "revert|undo|rv|undid", na=False
    ).astype(int)
    wiki.to_csv(WIKI_FILE, index=False)
 
wiki["timestamp"] = pd.to_datetime(wiki["timestamp"])
print(f"\n  ✅ Wiki revisions loaded: {len(wiki)}")
print(f"     Articles: {wiki['article'].value_counts().to_dict()}")

STEP 1: Fetching Wikipedia revisions
  📂 Found existing wiki_edits.csv — skipping fetch.

  ✅ Wiki revisions loaded: 4286
     Articles: {'Bitcoin': 1908, 'Ethereum': 1392, 'Cryptocurrency': 986}


In [17]:
# ============================================================
# STEP 2: FETCH BTC PRICES (Binance)
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Fetching BTC prices from Binance")
print("=" * 60)
 
def fetch_btc_prices():
    url = "https://api.binance.com/api/v3/klines"
    start_ts = int(pd.Timestamp("2019-01-01").timestamp() * 1000)
    end_ts   = int(pd.Timestamp("2023-12-31").timestamp() * 1000)
    all_candles = []
 
    while start_ts < end_ts:
        params = {"symbol": "BTCUSDT", "interval": "1d",
                  "startTime": start_ts, "endTime": end_ts, "limit": 1000}
        response = requests.get(url, params=params)
        data = response.json()
        if not data:
            break
        all_candles.extend(data)
        start_ts = data[-1][0] + 1
        time.sleep(0.3)
 
    df = pd.DataFrame(all_candles, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades",
        "taker_buy_base", "taker_buy_quote", "ignore"
    ])
    df["date"]   = pd.to_datetime(df["open_time"], unit="ms").dt.normalize()
    df["close"]  = df["close"].astype(float)
    df["volume"] = df["volume"].astype(float)
    df = df[["date", "close", "volume"]].copy()
    df["return"]     = df["close"].pct_change()
    df["volatility"] = df["return"].rolling(7).std()
    return df
 
if os.path.exists(BTC_FILE):
    print(f"  📂 Found existing {BTC_FILE} — skipping fetch.")
    btc = pd.read_csv(BTC_FILE)
else:
    btc = fetch_btc_prices()
    btc.to_csv(BTC_FILE, index=False)
 
btc["date"] = pd.to_datetime(btc["date"])
print(f"  ✅ BTC prices loaded: {len(btc)} days")
print(f"     Range: {btc['date'].min().date()} → {btc['date'].max().date()}")


STEP 2: Fetching BTC prices from Binance
  📂 Found existing btc_prices.csv — skipping fetch.
  ✅ BTC prices loaded: 1826 days
     Range: 2019-01-01 → 2023-12-31


In [18]:
# ============================================================
# STEP 3: TF-IDF ANALYSIS + AGGREGATION
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: TF-IDF Analysis")
print("=" * 60)
 
# Clean comments
wiki["comment"] = wiki["comment"].fillna("").str.lower()
wiki["comment"] = wiki["comment"].str.replace(r"[^a-z\s]", " ", regex=True)
wiki["comment"] = wiki["comment"].str.replace(r"\s+", " ", regex=True).str.strip()
 
# Aggregate per day
daily_comments = wiki.groupby("timestamp")["comment"].apply(
    lambda x: " ".join(x)
).reset_index()
daily_comments.columns = ["date", "daily_text"]
 
daily_stats = wiki.groupby("timestamp").agg(
    total_edits    = ("rev_id", "count"),
    unique_editors = ("user", "nunique"),
    revert_count   = ("is_revert", "sum"),
    anon_edits     = ("is_anon", "sum"),
).reset_index().rename(columns={"timestamp": "date"})
 
daily = daily_comments.merge(daily_stats, on="date")
 
# TF-IDF
vectorizer = TfidfVectorizer(
    max_features = 500,
    min_df       = 5,
    max_df       = 0.85,
    stop_words   = "english",
    ngram_range  = (1, 2),
)
tfidf_matrix = vectorizer.fit_transform(daily["daily_text"])
daily["tfidf_score"] = tfidf_matrix.mean(axis=1).A1
daily["tfidf_max"]   = np.array(tfidf_matrix.max(axis=1).todense()).flatten()
 
print(f"  ✅ Vocabulary: {len(vectorizer.get_feature_names_out())} terms")
print(f"     Active days: {len(daily)}")
 
# Merge with BTC
df = btc.merge(daily, on="date", how="left")
fill_cols = ["total_edits", "unique_editors", "revert_count", "tfidf_score", "tfidf_max"]
df[fill_cols] = df[fill_cols].fillna(0)
 
# Lags
df["tfidf_lag1"]      = df["tfidf_score"].shift(1)
df["tfidf_lag7"]      = df["tfidf_score"].shift(7)
df["tfidf_max_lag1"]  = df["tfidf_max"].shift(1)
df["volatility_lag1"] = df["volatility"].shift(1)
df["return_lag1"]     = df["return"].shift(1)
 
df.to_csv(FINAL_FILE, index=False)
print(f"     Final dataset: {len(df)} days → saved to {FINAL_FILE}")


STEP 3: TF-IDF Analysis
  ✅ Vocabulary: 500 terms
     Active days: 1085
     Final dataset: 1826 days → saved to wiki_tfidf_final.csv


In [19]:
# ============================================================
# STEP 4: CORRELATION ANALYSIS
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Correlation Analysis")
print("=" * 60)
 
df_clean = df.dropna(subset=["return", "volatility", "tfidf_lag1"]).copy()
print(f"  Observations: {len(df_clean)}")
 
x_vars = {
    "tfidf_score (same day)":  "tfidf_score",
    "tfidf_score (lag 1)":     "tfidf_lag1",
    "tfidf_score (lag 7)":     "tfidf_lag7",
    "tfidf_max (lag 1)":       "tfidf_max_lag1",
    "unique_editors":          "unique_editors",
    "total_edits":             "total_edits",
}
 
for target_name, target_col in [("BTC RETURN", "return"), ("BTC VOLATILITY", "volatility")]:
    df_t = df.dropna(subset=[target_col, "tfidf_lag1"]).copy()
    print(f"\n  --- vs {target_name} (n={len(df_t)}) ---")
    print(f"  {'Variable':<32} {'Pearson r':>10} {'p-val':>8} {'Spearman r':>11} {'p-val':>8}")
    print("  " + "=" * 65)
    for label, col in x_vars.items():
        pr, pp = stats.pearsonr(df_t[col], df_t[target_col])
        sr, sp = stats.spearmanr(df_t[col], df_t[target_col])
        sig_p = "***" if pp < 0.01 else ("**" if pp < 0.05 else ("*" if pp < 0.1 else ""))
        sig_s = "***" if sp < 0.01 else ("**" if sp < 0.05 else ("*" if sp < 0.1 else ""))
        print(f"  {label:<32} {pr:>+10.4f} {pp:>7.4f}{sig_p:<3} {sr:>+10.4f} {sp:>7.4f}{sig_s:<3}")
 
print("\n  Significance: * p<0.1  ** p<0.05  *** p<0.01")
 


STEP 4: Correlation Analysis
  Observations: 1819

  --- vs BTC RETURN (n=1825) ---
  Variable                          Pearson r    p-val  Spearman r    p-val
  tfidf_score (same day)              -0.0291  0.2140       -0.0129  0.5827   
  tfidf_score (lag 1)                 +0.0074  0.7536       +0.0028  0.9059   
  tfidf_score (lag 7)                    +nan     nan          +nan     nan   
  tfidf_max (lag 1)                   -0.0014  0.9514       +0.0029  0.9006   
  unique_editors                      -0.0154  0.5122       -0.0030  0.8984   
  total_edits                         +0.0014  0.9517       -0.0005  0.9813   

  --- vs BTC VOLATILITY (n=1819) ---
  Variable                          Pearson r    p-val  Spearman r    p-val
  tfidf_score (same day)              +0.0693  0.0031***    +0.1121  0.0000***
  tfidf_score (lag 1)                 +0.0720  0.0021***    +0.1146  0.0000***
  tfidf_score (lag 7)                 +0.0831  0.0004***    +0.0991  0.0000***
  tfidf_max (l

In [20]:
# ============================================================
# STEP 5: OLS REGRESSION
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: OLS Regression")
print("=" * 60)
 
df_ols = df.dropna(subset=[
    "volatility", "return", "tfidf_lag1",
    "tfidf_max_lag1", "volatility_lag1", "return_lag1"
]).copy()
print(f"  Observations: {len(df_ols)}")
 
y = df_ols["volatility"]
 
# Model 1: Simple
X1 = sm.add_constant(df_ols[["tfidf_lag1"]])
model1 = sm.OLS(y, X1).fit(cov_type="HC3")
 
# Model 2: Full multivariate with AR(1) control
X2 = sm.add_constant(df_ols[[
    "tfidf_lag1", "tfidf_max_lag1", "unique_editors",
    "total_edits", "volatility_lag1", "return_lag1"
]])
model2 = sm.OLS(y, X2).fit(cov_type="HC3")
 
# Model 3: Multivariate without AR(1) — shows direct TF-IDF effect
X3 = sm.add_constant(df_ols[[
    "tfidf_lag1", "tfidf_max_lag1", "unique_editors",
    "total_edits", "return_lag1"
]])
model3 = sm.OLS(y, X3).fit(cov_type="HC3")
 
# Model 4: Return prediction
y_ret = df_ols["return"]
X4 = sm.add_constant(df_ols[[
    "tfidf_lag1", "tfidf_max_lag1", "unique_editors",
    "total_edits", "return_lag1"
]])
model4 = sm.OLS(y_ret, X4).fit(cov_type="HC3")
 
# Summary Table
print("\n" + "=" * 75)
print("SUMMARY TABLE")
print("=" * 75)
print(f"  {'':28} {'Model 1':>10} {'Model 2':>10} {'Model 3':>10} {'Model 4':>10}")
print(f"  {'Dependent':28} {'Volat.':>10} {'Volat.':>10} {'Volat.':>10} {'Return':>10}")
print("  " + "-" * 70)
 
for v in ["tfidf_lag1", "tfidf_max_lag1", "unique_editors",
          "total_edits", "volatility_lag1", "return_lag1", "const"]:
    row = f"  {v:28}"
    for model in [model1, model2, model3, model4]:
        if v in model.params:
            coef = model.params[v]
            pval = model.pvalues[v]
            sig  = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
            row += f" {coef:>+8.4f}{sig:<2}"
        else:
            row += f" {'—':>10}"
    print(row)
 
print("  " + "-" * 70)
print(f"  {'R-squared':28} {model1.rsquared:>10.4f} {model2.rsquared:>10.4f} {model3.rsquared:>10.4f} {model4.rsquared:>10.4f}")
print(f"  {'Adj. R-squared':28} {model1.rsquared_adj:>10.4f} {model2.rsquared_adj:>10.4f} {model3.rsquared_adj:>10.4f} {model4.rsquared_adj:>10.4f}")
print(f"  {'Observations':28} {int(model1.nobs):>10} {int(model2.nobs):>10} {int(model3.nobs):>10} {int(model4.nobs):>10}")
print("  Significance: * p<0.1  ** p<0.05  *** p<0.01")
print("  Note: Robust standard errors (HC3)")
print("\n🎉 Pipeline complete!")
 


STEP 5: OLS Regression
  Observations: 1818

SUMMARY TABLE
                                  Model 1    Model 2    Model 3    Model 4
  Dependent                        Volat.     Volat.     Volat.     Return
  ----------------------------------------------------------------------
  tfidf_lag1                    +0.3351***  -0.0211    +0.2006*   +0.0939  
  tfidf_max_lag1                        —  +0.0013*   +0.0010    -0.0003  
  unique_editors                        —  +0.0001    +0.0013***  -0.0006  
  total_edits                           —  -0.0000    -0.0001    +0.0001  
  volatility_lag1                       —  +0.9102***          —          —
  return_lag1                           —  -0.0146*   -0.0114    -0.0714**
  const                         +0.0295***  +0.0023***  +0.0282***  +0.0024* 
  ----------------------------------------------------------------------
  R-squared                        0.0052     0.8318     0.0143     0.0056
  Adj. R-squared                   0.0